# **Tiki Book Aspect-based Sentiment Analysis (ABSA)**



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load Dataset

In [ ]:
import pandas as pd
with open(r'/content/drive/MyDrive/Group2_Final_AI/small_tiki_comment.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

header = lines[0].strip()  # Dòng đầu tiên trong file thường là tiêu đề (tên cột), ta lấy ra làm tên cột
lines = lines[1:]  # Lấy từ dòng thứ 2 trở đi (danh sách các bình luận)
df = pd.DataFrame(lines, columns=[header])


In [ ]:
df

In [ ]:
df.iloc[4].values

# Preprocessing

In [ ]:
import pandas as pd
import re
import string

# Đường dẫn tới file dữ liệu comment của Tiki
file_path = r"/content/drive/MyDrive/Group2_Final_AI/small_tiki_comment.txt"

# Đọc dữ liệu từ file CSV (định dạng .txt nhưng thực chất là CSV),
# bỏ qua các dòng lỗi format (on_bad_lines='skip')
df = pd.read_csv(file_path, sep=",", quotechar='"', on_bad_lines='skip', engine='python')

# Loại bỏ các dòng mà cột "content" bị thiếu (NaN) vì nội dung là chính, cần có để xử lý
df = df.dropna(subset=["content"])

# Các hàm chuẩn hóa dữ liệu text
# 1. Chuẩn hóa tiền tệ: Thay thế các số có đuôi k, m, b (ví dụ: 100k, 1.5m) thành từ "giá"
def normalize_money(sent):
    return re.sub(r'[0-9]+[.,0-9]*[kmb]', 'giá', sent, flags=re.IGNORECASE)

# 2. Chuẩn hóa hashtag: Thay thế các hashtag (#word) thành từ "tag"
def normalize_hastag(sent):
    return re.sub(r'#+\w+', 'tag', sent)

# 3. Chuẩn hóa website: Thay thế các đường link (URL) thành từ "website"
def normalize_website(sent):
    # Thay thế URL bắt đầu bằng http hoặc https
    result = re.sub(r'http[s]?://(?:[a-zA-Z0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', 'website', sent)
    # Thay thế các tên miền phổ biến (.com, .vn, .me) kèm theo đường dẫn thành "website"
    return re.sub(r'\w+(\.(com|vn|me))+((\/+([\.\w\_\-]+)?)+)?', 'website', result)

# 4. Xóa emoji trong câu bằng regex, loại bỏ các ký tự emoji Unicode phổ biến
def nomalize_emoji(sent):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u200d"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\u3030"
        u"\ufe0f"
        u"\u2764"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', sent)

# 5. Chuẩn hóa các từ viết tắt, teen code, biểu tượng thành từ chuẩn, ví dụ:
# "okie" thành "ok", "tks" thành "cám ơn", "⭐" thành "star", "kg" thành "không"...
def normalize_acronyms(sent):
    replace_list = {
        'ô kêi': ' ok ', 'okie': ' ok ', ' o kê ': ' ok ',
        'okey': ' ok ', 'ôkê': ' ok ', 'oki': ' ok ', ' oke ':  ' ok ',
        'okay':' ok ','okê':' ok ', ' tks ':' cám ơn ', 'thks':' cám ơn ',
        'thanks':' cám ơn ', 'ths':' cám ơn ', 'thank':' cám ơn ',
        '⭐':'star ', '*':'star ', '🌟':'star ', '🎉':' tích cực ',
        'kg ':' không ','not':' không ',' kh ':' không ','kô':' không ',
        'hok':' không ',' kp ':' không phải ',' ko ':' không ',' k ':' không ',
        'khong':' không ', 'he he':' tích cực ','hehe':' tích cực ',
        'hihi':' tích cực ', 'haha':' tích cực ', 'hjhj':' tích cực ',
        ' lol ':' tiêu cực ',' cc ':' tiêu cực ','cute':' dễ thương ',
        'huhu':' tiêu cực ', ' vs ':' với ', 'wa':' quá ', 'wá':' quá',
        'j':' gì ', 'sz ':' cỡ ', 'size':' cỡ ', 'đx ':' được ',
        'dk':' được ', 'dc':' được ', 'đk':' được ', 'đc':' được ',
        'authentic':' chuẩn chính hãng ','auth ':' chuẩn chính hãng ',
        'thick':' tích cực ', 'store':' cửa hàng ', 'shop':' cửa hàng ',
        'sp':' sản phẩm ', 'gud':' tốt ','god':' tốt ','wel done':' tốt ',
        'good':' tốt ', 'sấu':' xấu ','gut':' tốt ', ' tot ':' tốt ',
        ' nice ':' tốt ', 'perfect':'rất tốt', 'bt':' bình thường ',
        'time':' thời gian ', 'qá':' quá ', ' ship ':' giao hàng ',
        ' m ':' mình ', ' mik ':' mình ', 'ể':'ể', 'product':'sản phẩm',
        'quality':'chất lượng','chat':' chất ', 'excelent':'hoàn hảo',
        'bad':'tệ','fresh':' tươi ','sad':' tệ ', 'date':' hạn sử dụng ',
        'hsd':' hạn sử dụng ','quickly':' nhanh ', 'quick':' nhanh ',
        'fast':' nhanh ','delivery':' giao hàng ',' síp ':' giao hàng ',
        'beautiful':' đẹp tuyệt vời ', ' tl ':' trả lời ', ' r ':' rồi ',
        ' shopE ':' cửa hàng ',' order ':' đặt hàng ', 'chất lg':' chất lượng ',
        ' sd ':' sử dụng ',' dt ':' điện thoại ',' nt ':' nhắn tin ',
        ' tl ':' trả lời ',' sài ':' xài ','bjo':' bao giờ ','thik':' thích ',
        ' sop ':' cửa hàng ', ' fb ':' facebook ', ' face ':' facebook ',
        ' very ':' rất ','quả ng ':' quảng  ','dep':' đẹp ',' xau ':' xấu ',
        'delicious':' ngon ','hàg':' hàng ','qủa':' quả ','iu':' yêu ',
        'fake':' giả mạo ', 'trl':'trả lời', '><':' tích cực ',
        ' por ':' tệ ',' poor ':' tệ ', 'ib':' nhắn tin ', 'rep':' trả lời ',
        'fback':' feedback ','fedback':' feedback '
    }
    text = sent
    # Thay thế từng từ viết tắt hoặc teen code theo dictionary trên
    for k, v in replace_list.items():
        text = text.replace(k, v)
    return text

# 6. Hàm tổng hợp chuẩn hóa:
# Gọi lần lượt các hàm trên để chuẩn hóa một câu comment
def normalize(sent):
    result = normalize_money(sent)       # Chuẩn hóa tiền
    result = normalize_hastag(result)   # Chuẩn hóa hashtag
    result = normalize_website(result)  # Chuẩn hóa website
    result = nomalize_emoji(result)     # Loại bỏ emoji
    result = normalize_acronyms(result) # Chuẩn hóa từ viết tắt, teen code
    result = result.lower()              # Chuyển về chữ thường toàn bộ
    # Loại bỏ dấu câu (thay tất cả dấu câu thành khoảng trắng)
    result = result.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    # Thay thế nhiều khoảng trắng liên tiếp bằng 1 khoảng trắng duy nhất và loại bỏ khoảng trắng thừa 2 đầu
    result = re.sub(r'\s+', ' ', result).strip()
    return result
df['content_normalized'] = df['content'].apply(normalize)
print(df[['content', 'content_normalized']].head())
df.to_csv('/content/drive/MyDrive/Group2_Final_AI/small_tiki_comment_normalized.csv', index=False)

In [ ]:
!pip install underthesea sentence-transformers


# Aspects Extraction

In [ ]:
import re
import os
import torch
import pandas as pd
from underthesea import word_tokenize, sent_tokenize
from sentence_transformers import SentenceTransformer, util

# === Load Sentence-BERT model tiếng Việt ===
# Tải mô hình Sentence-BERT huấn luyện sẵn cho tiếng Việt để chuyển câu thành vector embedding
model = SentenceTransformer('VoVanPhuc/sup-SimCSE-VietNamese-phobert-base')
# Simple Contrastive Learning of Sentence Embeddings là một phương pháp học biểu diễn câu (sentence embeddings) rất hiệu quả và đơn giản
# Không tối ưu để train phân loại sentiment, vì nó tập trung học embedding câu chứ không học phân biệt các nhãn sentiment

# === Khai báo các khía cạnh cần trích xuất ===
# Dùng dictionary để map mã khía cạnh sang câu mô tả tiếng Việt tương ứng
aspect_to_vietnamese = {
    "BOOK#GENERAL": "sách nói chung",
    "BOOK#PRICE": "giá sách",
    "BOOK#QUALITY": "chất lượng sách",
    "BOOK#CONTENT": "nội dung sách",
    "BOOK#FORMAT": "hình thức sách",
    "BOOK#READER_EXPERIENCE": "trải nghiệm người đọc",
    "BOOK#RECOMMENDATION": "khuyến nghị sách",
    "DELIVERY#SERVICE": "dịch vụ giao hàng",
    "SELLER#SERVICE": "dịch vụ người bán"
}

# Tách riêng danh sách mã khía cạnh và các câu mô tả
aspect_keys = list(aspect_to_vietnamese.keys())
aspect_texts = list(aspect_to_vietnamese.values())

# Tạo embedding cho các câu mô tả khía cạnh (vector số biểu diễn ngữ nghĩa)
aspect_embeddings = model.encode(aspect_texts, convert_to_tensor=True)

# === Hàm làm sạch và chuẩn hóa văn bản ===
def clean_text(text):
    text = text.lower()  # chuyển về chữ thường
    # loại bỏ ký tự không phải chữ cái, số, dấu cách, dấu câu hợp lệ, giữ nguyên dấu tiếng Việt
    text = re.sub(
        r'[^\w\sáàảãạăắằẳẵặâấầẩẫậđéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợ'
        r'úùủũụưứừửữựýỳỷỹỵ!.?,;:\'\"()\[\] ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()  # thay nhiều khoảng trắng thành 1 và bỏ khoảng trắng đầu cuối
    return text

# Hàm tách câu từ đoạn văn bản (dùng thư viện underthesea)
def split_sentences(text):
    try:
        return sent_tokenize(text)
    except:
        # Nếu lỗi thì trả về nguyên đoạn văn bản như 1 câu duy nhất
        return [text]

# === Hàm trích xuất khía cạnh từ văn bản ===
def extract_aspects(text, threshold=0.3, top_k=2):
    aspects = set()  # Tập các khía cạnh đã phát hiện (tránh trùng)
    sentences = split_sentences(clean_text(text))  # Làm sạch rồi tách câu
    for sent in sentences:
        if not sent.strip():
            continue  # bỏ qua câu rỗng
        try:
            # Tách từ câu theo chuẩn tiếng Việt, đầu ra là string các từ cách nhau khoảng trắng
            tokenized_sent = word_tokenize(sent, format="text")
            # Tạo embedding cho câu vừa tách từ
            sent_embedding = model.encode(tokenized_sent, convert_to_tensor=True)
            # Tính độ tương đồng cosine giữa câu và từng khía cạnh
            scores = util.cos_sim(sent_embedding, aspect_embeddings)[0]
            # Lấy chỉ số của top_k khía cạnh có similarity cao nhất
            top_idxs = torch.topk(scores, k=top_k).indices
            for i in top_idxs:
                # Nếu similarity trên ngưỡng threshold thì thêm khía cạnh đó vào tập kết quả
                if scores[i] >= threshold:
                    aspects.add(aspect_keys[i])
        except Exception as e:
            print(f"Error: {sent} - {e}")  # In lỗi nếu có
            continue
    # Trả về danh sách các khía cạnh phát hiện được
    return list(aspects)

# === Đọc dữ liệu đã chuẩn hóa từ file CSV ===
input_path = '/content/drive/MyDrive/Group2_Final_AI/small_tiki_comment_normalized.csv'
df = pd.read_csv(input_path)

# === Áp dụng hàm trích xuất khía cạnh cho từng comment ===
df['detected_aspects'] = df['content'].apply(lambda x: extract_aspects(str(x)))

# === Tách mỗi khía cạnh ra thành 1 dòng riêng biệt ===
df_aspect = df.explode('detected_aspects').rename(columns={'detected_aspects': 'aspect'})

# Loại bỏ các dòng không phát hiện khía cạnh nào (NaN)
df_aspect = df_aspect.dropna(subset=['aspect'])

# === Bỏ cột comment gốc để giảm dung lượng file lưu ===
df_aspect = df_aspect.drop(columns=['content'])

# === Lưu kết quả ra file CSV mới ===
output_path = '/content/drive/MyDrive/Group2_Final_AI/Tiki_aspect_extraction.csv'
df_aspect.to_csv(output_path, index=False)

# === In 10 dòng đầu để kiểm tra kết quả ===
print(df_aspect.head(10))


# Sentiment Labelling

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# === 1. Load model và tokenizer PhoBERT sentiment ===
def load_sentiment_model(checkpoint="mr4/phobert-base-vi-sentiment-analysis"): # Model đã được huấn luyện kỹ trên tập dữ liệu sentiment tiếng Việt, nên hiểu rõ cách biểu đạt cảm xúc trong ngôn ngữ này.
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Không cập nhật trọng số khi dự đoán
    return tokenizer, model, device

# === 2. Hàm đọc dữ liệu đã extract aspect từ CSV ===
def load_aspect_data(filepath, text_col="content_normalized"):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['aspect'])  # Loại dòng không có khía cạnh
    return df, text_col

# === 3. Hàm chuẩn bị dữ liệu đầu vào cho sentiment analysis ===
def prepare_sentiment_input(df, text_col="content_normalized"):
    combined_texts = []
    orig_texts = []
    aspects = []

    for _, row in df.iterrows():
        text = str(row[text_col]).strip()
        aspect = str(row['aspect']).strip()
        if text and aspect:
            combined_text = f"content: {text} | aspect: {aspect}"
            combined_texts.append(combined_text)
            orig_texts.append(text)
            aspects.append(aspect)
    return combined_texts, orig_texts, aspects

# === 4. Hàm dự đoán cảm xúc theo batch ===
def predict_sentiment_batch(texts, orig_texts, aspects, tokenizer, model, device, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_orig = orig_texts[i:i+batch_size]
        batch_aspect = aspects[i:i+batch_size]

        # Mã hóa và chuyển tensor sang thiết bị
        encoded = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            probs = F.softmax(outputs.logits, dim=-1)

        for j, prob in enumerate(probs):
            label = torch.argmax(prob).item()
            results.append({
                "content_normalized": batch_orig[j],
                "aspect": batch_aspect[j],
                "sentiment_label": label,  # 0: negative, 1: neutral, 2: positive
                "score_negative": prob[0].item(),
                "score_neutral": prob[1].item(),
                "score_positive": prob[2].item(),
            })
    return results

# === 5. Hàm chính để chạy toàn bộ quy trình ===
def run_sentiment_pipeline(input_path, output_path):
    # Bước 1: Load model
    tokenizer, model, device = load_sentiment_model()

    # Bước 2: Load dữ liệu đã extract aspect
    df, text_col = load_aspect_data(input_path)

    # Bước 3: Chuẩn bị dữ liệu đầu vào
    combined_texts, orig_texts, aspects = prepare_sentiment_input(df, text_col)

    # Bước 4: Dự đoán sentiment
    results = predict_sentiment_batch(combined_texts, orig_texts, aspects, tokenizer, model, device)

    # Bước 5: Lưu kết quả
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path, index=False, encoding='utf-8-sig', float_format="%.4f")

    print(results_df.head())

input_file = "/content/drive/MyDrive/Group2_Final_AI/Tiki_aspect_extraction.csv"
output_file = "/content/drive/MyDrive/Group2_Final_AI/Tiki_books_aspect_sentiment_labeled_1.csv"
run_sentiment_pipeline(input_file, output_file)


In [ ]:
import pandas as pd

# Đọc lại kết quả đã lưu
df = pd.read_csv("/content/drive/MyDrive/Group2_Final_AI/Tiki_books_aspect_sentiment_labeled_1.csv")

# Đếm số lượng cảm xúc theo từng aspect
summary = df.groupby(["aspect", "sentiment_label"]).size().unstack(fill_value=0)
# .unstack(fill_value=0)
# Biến index thứ hai (sentiment_label) thành cột trong DataFrame mới.
# Những nhóm không có dữ liệu sẽ được điền giá trị là 0 (thay vì NaN).

# Đổi tên cột để hiển thị nhãn rõ ràng hơn
summary.columns = ['0: negative', '1: neutral', '2: positive']
print(summary)


In [ ]:
import matplotlib.pyplot as plt

# Đếm số lần xuất hiện của từng aspect
aspect_counts = df['aspect'].value_counts()

# Vẽ biểu đồ bar
plt.figure(figsize=(10, 6))
aspect_counts.plot(kind='bar', color='green') # red, blue, orange
plt.title("Aspect Distribution")
plt.xlabel("Aspect")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right') # Xoay nhãn trục X 45 độ, căn phải cho dễ đọc
plt.tight_layout() # Tự động căn chỉnh biểu đồ cho đẹp
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot stacked bar chart
summary.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="viridis") # 'plasma', 'cividis', 'magma'
plt.title("Sentiments Distibution by Aspects")
plt.xlabel("Aspects")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Sentiment") # Hiển thị chú thích, tiêu đề chú thích là "Sentiment"
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Đếm tần suất của các nhãn sentiment trong cột 'sentiment_label'
sentiment_counts = df['sentiment_label'].value_counts()

# Tạo biểu đồ tròn (pie chart) với kích thước 7x7 inch
plt.figure(figsize=(7, 7))

# Vẽ pie chart
plt.pie(
    sentiment_counts,                   # Kích thước từng phần trong biểu đồ
    labels=sentiment_counts.index,     # Nhãn cho từng phần (ví dụ: positive, negative, neutral)
    autopct='%1.1f%%',                 # Hiển thị phần trăm với 1 chữ số thập phân
    startangle=90,                     # Bắt đầu vẽ từ góc 90 độ (đỉnh)
    colors=['lightgreen', 'lightyellow', 'lightcoral']  # Màu sáng cho từng phần   , colors = ['lightblue', 'lightyellow', 'lightcoral']

)

plt.title("Sentiment Distribution")    # Tiêu đề biểu đồ
plt.show()                            # Hiển thị biểu đồ

# Training Preparation

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Group2_Final_AI/Tiki_books_aspect_sentiment_labeled_1.csv')
df

In [ ]:
df_train = df.drop(columns=['score_positive', 'score_neutral', 'score_negative'])


In [ ]:
df_train.to_csv('/content/drive/MyDrive/Group2_Final_AI/train_data.csv', index=False) # index = false --> bỏ qua cột index
df_subset = df.head(10000)
df_subset.to_csv('/content/drive/MyDrive/Group2_Final_AI/train_data_subset.csv', index=False)

# Train-Test Split

In [ ]:
!pip install iterative-stratification

In [ ]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit  # Thư viện hỗ trợ chia dữ liệu theo nhãn đa lớp

# Bước 1: Đọc dữ liệu từ file CSV chứa các dòng đã gán nhãn (content, aspect, sentiment)
df = pd.read_csv('/content/drive/MyDrive/Group2_Final_AI/train_data_subset.csv')

# Bước 2: Xóa các dòng bị thiếu dữ liệu ở các cột cần thiết
df = df.dropna(subset=["content_normalized", "aspect", "sentiment_label"])

# Bước 3: Giữ lại các dòng có sentiment_label là 0 (tiêu cực), 1 (trung lập), 2 (tích cực)
df = df[df["sentiment_label"].isin([0, 1, 2])]

# Bước 4: Tạo ma trận biểu diễn tỉ lệ các nhãn cảm xúc (0, 1, 2) cho mỗi câu 'content_normalized'
# Mỗi câu sẽ được biểu diễn bằng vector gồm tỉ lệ xuất hiện của các nhãn sentiment trong các dòng tương ứng
label_vectors = df.groupby('content_normalized')['sentiment_label'] \
                 .apply(lambda x: x.value_counts(normalize=True)) \
                 .unstack(fill_value=0)

# Bước 5: Dùng MultilabelStratifiedShuffleSplit để chia train/test theo câu --> Giữ nguyên tỷ lệ phân phối các nhãn đa nhãn trong cả tập train và test:
# Bảo đảm mỗi tập có phân phối các nhãn cảm xúc tương đồng
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(msss.split(label_vectors, label_vectors))

# Lấy danh sách nội dung câu tương ứng với train/test
train_contents = label_vectors.index[train_idx]
test_contents = label_vectors.index[test_idx]

# Bước 6: Lọc lại dữ liệu gốc dựa trên các câu đã được chia
# Giúp đảm bảo nội dung giữa train và test không bị trùng
train_df = df[df['content_normalized'].isin(train_contents)].reset_index(drop=True)
test_df = df[df['content_normalized'].isin(test_contents)].reset_index(drop=True)

# Bước 7: Tạo câu đầu vào dạng "content: ... | aspect: ..." để đưa vào mô hình học
train_texts = train_df.apply(lambda row: f"content: {row['content_normalized']} | aspect: {row['aspect']}", axis=1).tolist()
test_texts = test_df.apply(lambda row: f"content: {row['content_normalized']} | aspect: {row['aspect']}", axis=1).tolist()

# Bước 8: Lấy nhãn cảm xúc cho từng dòng
train_labels = train_df['sentiment_label'].tolist()
test_labels = test_df['sentiment_label'].tolist()

# Bước 9: In thông tin tổng quan để kiểm tra kích thước và phân phối nhãn
print(f"Train samples: {len(train_texts)}, Test samples: {len(test_texts)}")

# In tỉ lệ phần trăm từng nhãn trong tập train và test
print("Class distribution in train:", pd.Series(train_labels).value_counts(normalize=True).to_dict())
print("Class distribution in test :", pd.Series(test_labels).value_counts(normalize=True).to_dict())

# In số lượng mẫu thực tế cho từng nhãn trong train và test
print("Class counts in train:", pd.Series(train_labels).value_counts().to_dict())
print("Class counts in test :", pd.Series(test_labels).value_counts().to_dict())

In [ ]:
train_df

# Solve Imbalance using Class Weight

In [ ]:
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Tính trọng số cân bằng cho từng lớp để xử lý dữ liệu mất cân bằng
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
alpha = torch.tensor(class_weights, dtype=torch.float) # Chuyển thành tensor để dùng trong PyTorch

# Định nghĩa hàm mất mát Focal Loss để tập trung vào các mẫu khó phân loại
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'): #khi tạo một đối tượng FocalLoss(...), hàm này được gọi đầu tiên để thiết lập các tham số và trạng thái cho đối tượng đó.
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

def forward(self, inputs, targets):
    # Tính loss cross entropy từng mẫu (không lấy trung bình)
    ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')

    # Lấy trọng số lớp cho từng mẫu (nếu có), nếu không thì để 1
    if self.alpha is not None:
        weights = self.alpha[targets].to(targets.device)
    else:
        weights = 1.0

    # Tính xác suất dự đoán đúng (pt)
    pt = torch.exp(-ce_loss)

    # Tính focal loss: trọng số * (1 - pt)^gamma * cross entropy
    focal_loss = weights * (1 - pt) ** self.gamma * ce_loss

    # Trả về trung bình hoặc tổng loss trên batch
    if self.reduction == 'mean':
        return focal_loss.mean()
    else:
        return focal_loss.sum()

# 3. Tạo loss_fn để dùng trong Trainer
loss_fn = FocalLoss(alpha=alpha, gamma=2.0)


# Sentiment Classification using VinAI-Phobert

In [ ]:
!pip install --upgrade transformers

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# 1. Định nghĩa Dataset class để xử lý dữ liệu đầu vào cho mô hình
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts                # Danh sách câu (content + aspect)
        self.labels = labels              # Nhãn tương ứng (sentiment)
        self.tokenizer = tokenizer        # Tokenizer để chuyển câu thành token
        self.max_length = max_length      # Độ dài tối đa cho mỗi câu

    def __len__(self):
        return len(self.texts)            # Trả về số lượng mẫu trong dataset

    def __getitem__(self, idx):
        text = self.texts[idx]            # Lấy câu thứ idx
        label = self.labels[idx]          # Lấy nhãn tương ứng
        encoding = self.tokenizer(        # Tokenize câu
            text,
            max_length=self.max_length,
            padding='max_length',         # Padding hoặc cắt câu về độ dài max_length
            truncation=True,
            return_tensors='pt',          # Trả về tensor PyTorch
        )
        # Đổi từ batch size 1 về dạng 1D tensor để phù hợp với mô hình
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = torch.tensor(label)   # Thêm nhãn vào tensor
        return item

# 2. Khởi tạo tokenizer từ model PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# Tạo dataset train và test với tokenizer và dữ liệu đã chuẩn bị sẵn
train_dataset = TextDataset(train_texts, train_labels, tokenizer)
test_dataset = TextDataset(test_texts, test_labels, tokenizer)

# 3. Load model PhoBERT để fine-tune cho bài toán phân loại sentiment với 3 nhãn
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=3)

# Lý do chọn model gốc "vinai/phobert-base":
# - Chưa bị fine-tune hay ảnh hưởng bởi task khác
# - Khi fine-tune trên dữ liệu sentiment của bạn sẽ học đặc trưng riêng
# - Giúp tăng độ chính xác và phù hợp với dữ liệu của bạn

# 4. Hàm tính toán các metric để đánh giá model
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)   # Lấy nhãn dự đoán từ logits

    # Tính các chỉ số Precision, Recall, F1 score theo weighted average (tính trung bình có trọng số)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, preds)  # Độ chính xác
    conf_matrix = confusion_matrix(labels, preds)  # Ma trận nhầm lẫn

    # Trả về dictionary các chỉ số đánh giá
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": conf_matrix
    }

# 5. Định nghĩa FocalLoss để cải thiện xử lý dữ liệu mất cân bằng
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha      # Trọng số class (cân bằng dữ liệu)
        self.gamma = gamma      # Tham số điều chỉnh tập trung vào mẫu khó
        self.reduction = reduction  # Cách trả về loss ('mean' hoặc 'sum')

    def forward(self, inputs, targets):
        # Tính cross entropy loss từng mẫu, không lấy trung bình
        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction='none')

        # Nếu có trọng số alpha, gán trọng số từng mẫu theo lớp
        if self.alpha is not None:
            alpha = self.alpha.to(targets.device)  # Chuyển alpha về device phù hợp (CPU/GPU)
            at = alpha[targets]
        else:
            at = 1.0  # Nếu không có alpha, dùng trọng số mặc định 1

        pt = torch.exp(-ce_loss)   # pt là xác suất dự đoán đúng cho từng mẫu
        loss = at * (1 - pt) ** self.gamma * ce_loss  # Công thức focal loss

        # Trả về loss trung bình hoặc tổng trên batch
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# 6. Tính trọng số cân bằng class từ dữ liệu train
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
alpha = torch.tensor(class_weights, dtype=torch.float)  # Chuyển sang tensor để dùng trong loss

# Khởi tạo hàm loss focal loss với alpha và gamma=2.0
loss_fn = FocalLoss(alpha=alpha, gamma=2.0)

# 7. Định nghĩa CustomTrainer để dùng hàm loss focal thay cho loss mặc định
class CustomTrainer(Trainer):
    def __init__(self, loss_fn, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")  # Lấy nhãn ra khỏi inputs
        outputs = model(**inputs)       # Forward mô hình
        logits = outputs.get("logits")  # Lấy logits dự đoán
        loss = self.loss_fn(logits, labels)  # Tính focal loss với logits và labels
        return (loss, outputs) if return_outputs else loss

# 8. Cài đặt các tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./results",              # Thư mục lưu kết quả
    # overwrite_output_dir=True,            # Ghi đè thư mục nếu tồn tại
    num_train_epochs=5,                   # Số epoch huấn luyện
    per_device_train_batch_size=8,       # Batch size khi train
    per_device_eval_batch_size=8,        # Batch size khi eval
    learning_rate=2e-5,                   # Learning rate
    weight_decay=0.01,                   # L2 regularization thêm vào loss một phần tỷ lệ với tổng bình phương các trọng số
    warmup_steps=300,                    # Số bước warmup learning rate
    lr_scheduler_type="cosine",          # Scheduler learning rate
    gradient_accumulation_steps=2,       # Tổng gradient qua mấy batch mới cập nhật
    fp16=True,                          # Dùng float16 tăng tốc (GPU có hỗ trợ)
    logging_strategy="steps",             # Ghi log theo bước
    logging_steps=100,                    # Mỗi 100 bước ghi log 1 lần
    save_total_limit=2,                   # Giới hạn số model lưu
    report_to="none",                    # Không gửi log lên nền tảng nào
    disable_tqdm=False                   # Hiện thanh tiến trình
)

# 9. Khởi tạo Trainer với model, dữ liệu, loss, metric, tham số huấn luyện
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    loss_fn=loss_fn,
)

# Bắt đầu huấn luyện
trainer.train()


# Model Evaluation

In [ ]:
import pandas as pd
import torch.nn.functional as F  # Đảm bảo đã import F từ torch.nn.functional

# 10. Dự đoán trên tập test
predictions = trainer.predict(test_dataset)

# Lấy ra kết quả nhãn dự đoán và xác suất
pred_labels = np.argmax(predictions.predictions, axis=1)
probs = predictions.predictions
probs = F.softmax(torch.tensor(probs), dim=1).numpy()

# Tạo DataFrame chứa kết quả
results_df = pd.DataFrame({
    'true_label': predictions.label_ids,
    'pred_label': pred_labels,
    'prob_0': probs[:, 0],
    'prob_1': probs[:, 1],
    'prob_2': probs[:, 2],
})

# Hiển thị 5 dòng đầu tiên
print(results_df.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
eval_results = trainer.evaluate()

# Hiển thị bảng kết quả
eval_df = pd.DataFrame([{
    'Loss': eval_results['eval_loss'],
    'Accuracy': eval_results['eval_accuracy'],
    'Precision': eval_results['eval_precision'],
    'Recall': eval_results['eval_recall'],
    'F1': eval_results['eval_f1'],
}])
print(eval_df)